In [1]:
from pathlib import Path
import sys

# Add the current working directory to the Python path to allow local imports
sys.path.insert(0, str(Path.cwd()))

from kbprojection import SNLILoader
import kbprojection.data as data

snli_loader = SNLILoader()
snli_loader.load(splits=["dev"])  # first run downloads SNLI (needs network)


[SNLILoader] Loaded 9842 problems from dev.


### Download the SNLI to a path

In [8]:
from pathlib import Path

import kbprojection.downloads as downloads
from kbprojection.loaders.snli import SNLILoader

# SNLI is distributed as a zip (no official standalone snli_1.0_dev.jsonl URL on Stanford’s site).
data_dir = Path.cwd() / "data" / "snli"
zip_path = data_dir / "snli_1.0.zip"
downloads.download_file(url=SNLILoader.URL, destination=zip_path)
downloads.extract_zip(zip_path, data_dir)
# After extract, dev is at: data_dir / "snli_1.0" / "snli_1.0_dev.jsonl"

[Download] Downloading https://nlp.stanford.edu/projects/snli/snli_1.0.zip to /Users/jorrytdejong/Documents/RNL paper publication/data/snli/snli_1.0.zip...
[Download] Finished: /Users/jorrytdejong/Documents/RNL paper publication/data/snli/snli_1.0.zip
[Extract] Extracting /Users/jorrytdejong/Documents/RNL paper publication/data/snli/snli_1.0.zip to /Users/jorrytdejong/Documents/RNL paper publication/data/snli...
[Extract] Skipping invalid filename: snli_1.0/Icon
[Extract] Finished.


In [2]:
data.get_snli_problem(snli_loader, key='386160015.jpg#3r1e')

{'annotator_labels': ['entailment',
  'entailment',
  'entailment',
  'entailment',
  'entailment'],
 'captionID': '386160015.jpg#3',
 'gold_label': 'entailment',
 'pairID': '386160015.jpg#3r1e',
 'sentence1': 'Two dogs playing on snow covered ground',
 'sentence1_binary_parse': '( ( ( Two dogs ) ( playing ( on snow ) ) ) ( covered ground ) )',
 'sentence1_parse': '(ROOT (S (NP (NP (CD Two) (NNS dogs)) (VP (VBG playing) (PP (IN on) (NP (NN snow))))) (VP (VBD covered) (NP (NN ground)))))',
 'sentence2': 'Dogs are playing on snow.',
 'sentence2_binary_parse': '( Dogs ( ( are ( playing ( on snow ) ) ) . ) )',
 'sentence2_parse': '(ROOT (S (NP (NNS Dogs)) (VP (VBP are) (VP (VBG playing) (PP (IN on) (NP (NN snow))))) (. .)))'}

In [3]:
data.random_snli_key(snli_loader)

'489723654.jpg#0r1e'

In [11]:
from kbprojection.downloads import check_nltk


check_nltk(package='punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/jorrytdejong/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Filtering.py

In [12]:
import kbprojection.filtering as filtering

filtering.get_lemmatizer()

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/jorrytdejong/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


<WordNetLemmatizer>

In [13]:
filtering.get_st_model()

/Users/jorrytdejong/Documents/RNL paper publication/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [15]:
filtering.tokenize('tokenize this!')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/jorrytdejong/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


['tokenize', 'this', '!']

In [16]:
filtering.remove_underscores('remove_underscores_test')

'remove underscores test'

In [21]:
filtering.normalize_kb_args('isa(apple_tree, tree)')

'isa(apple tree, tree)'

In [22]:
filtering.drop_leading_preposition('on the table')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/jorrytdejong/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


'the table'

In [26]:
filtering.parse_kb_injection('isa(apple_tree, tree)')

('isa', 'apple_tree', 'tree')

In [27]:
filtering.create_rel(pred='is_a', arg1='apple_tree', arg2='tree')

'is_a(apple_tree, tree)'

In [29]:
# Example with a single string:
single_premise = "A man is playing guitar"
normalized_single = filtering.normalize_premises(single_premise)
print(normalized_single)

# Example with a list of strings:
premise_list = ["A man is playing guitar"]
normalized_list = filtering.normalize_premises(premise_list)
print(normalized_list)

A man is playing guitar
A man is playing guitar


In [32]:
parsed_injection = filtering.parse_kb_injection('isa(apple_tree, tree)')
print(parsed_injection)
predicate, arg1, arg2 = parsed_injection
original_text = 'isa(apple_tree, tree)'
candidates = filtering.generate_all_candidates(predicate, arg1, arg2, original_text)
for cand in candidates:
    print(cand)

('isa', 'apple_tree', 'tree')
isa(apple tree, tree)
isa(tree, apple tree)


## LLM

In [41]:
from kbprojection.llm import GenericAIClient

client = GenericAIClient()

# detect provider
client._detect_provider()
client._setup_client()
client.generate(prompt='Hello, world!', model='gpt-4o-mini')

'Hello! How can I assist you today?'

In [43]:
import kbprojection.llm as llm

llm.extract_kb_from_output(
    """
    [KB_START]
    isa_wn(dog, animal)
    disj(sit, stand)
    [KB_END]
    The dog is either sitting or standing.
    """
)

llm.extract_kb_from_output(
    """
    isa_wn(dog, animal)
    disj(sit, stand)
    The dog is either sitting or standing.
    """
)

['isa_wn(dog, animal)', 'disj(sit, stand)']

In [65]:
from kbprojection.loaders.snli import SNLILoader
from kbprojection import llm

loader = SNLILoader(data_dir="./data/snli")
prob = loader.get_problem("386160015.jpg#3r1e", split="dev")
print(prob)

llm.call_llm(
    provider="openai",
    model="gpt-4o-mini",
    prompt_style="legacy_icl",
    prob=prob,
)


[SNLILoader] Loaded 9842 problems from dev.
id='386160015.jpg#3r1e' premises=['Two dogs playing on snow covered ground'] hypothesis='Dogs are playing on snow.' gold_label=<NLILabel.ENTAILMENT: 'entailment'> dataset='snli' split='dev' original_data={'annotator_labels': ['entailment', 'entailment', 'entailment', 'entailment', 'entailment'], 'captionID': '386160015.jpg#3', 'gold_label': 'entailment', 'pairID': '386160015.jpg#3r1e', 'sentence1': 'Two dogs playing on snow covered ground', 'sentence1_binary_parse': '( ( ( Two dogs ) ( playing ( on snow ) ) ) ( covered ground ) )', 'sentence1_parse': '(ROOT (S (NP (NP (CD Two) (NNS dogs)) (VP (VBG playing) (PP (IN on) (NP (NN snow))))) (VP (VBD covered) (NP (NN ground)))))', 'sentence2': 'Dogs are playing on snow.', 'sentence2_binary_parse': '( Dogs ( ( are ( playing ( on snow ) ) ) . ) )', 'sentence2_parse': '(ROOT (S (NP (NNS Dogs)) (VP (VBP are) (VP (VBG playing) (PP (IN on) (NP (NN snow))))) (. .)))'}


['isa_wn(dog, dogs)', 'isa_wn(snow, outdoors)']

In [63]:

llm.inject_kb_for_example(prob=prob, model="gpt-4o-mini", prompt_style="legacy_icl")


['isa_wn(dog, dog)', 'isa_wn(ground, snow)']

# Models

In [72]:
from kbprojection.models import NLIProblem, NLILabel

# This line is attempting to cast the NLILabel class to a list, but the syntax is incorrect.
# The correct way to get all members of an Enum (like NLILabel) as a list is:
print(list(NLILabel))

# Explanation:
# NLILabel is likely an Enum class representing possible NLI (Natural Language Inference) labels, such as 'entailment', 'neutral', or 'contradiction'.
# Using list(NLILabel) returns a list of all enumeration members of NLILabel.

# Enum classes, short for "enumeration" classes, are a special data type in Python (and many other languages)
# that allow you to define a set of named, constant, and unique values. Enumerations are commonly used to 
# represent a fixed set of possible options, such as days of the week, directions, or labels in a classification task.

# In Python, enums are created by subclassing the `Enum` base class from the `enum` module.
# Each member of the enum has a name and a value. Enum members are immutable and unique.
# Example:

from enum import Enum

class Color(Enum):
    RED = 1
    GREEN = 2
    BLUE = 3

# Usage example:
print(Color.RED)        # Color.RED
print(Color.RED.name)   # RED
print(Color.RED.value)  # 1
print(list(Color))      # [<Color.RED: 1>, <Color.GREEN: 2>, <Color.BLUE: 3>]

# Enum classes help make code more readable, less error-prone, and self-documenting,
# by providing meaningful names instead of just using literal values.

[<NLILabel.ENTAILMENT: 'entailment'>, <NLILabel.CONTRADICTION: 'contradiction'>, <NLILabel.NEUTRAL: 'neutral'>, <NLILabel.UNKNOWN: '-'>]
Color.RED
RED
1
[<Color.RED: 1>, <Color.GREEN: 2>, <Color.BLUE: 3>]


In [73]:
raw = prob

In [78]:
from kbprojection.models import NLIProblem, NLILabel

problem = NLIProblem(
    id="386160015.jpg#3r1e",
    premises=["Two dogs playing on snow covered ground"],
    hypothesis="Dogs are playing on snow.",
    gold_label=NLILabel.ENTAILMENT,
    dataset="snli",
    split="dev",
)


# Orchestration

In [ ]:
import kbprojection.orchestration as orchestration

output = orchestration.process_single_problem(
    prob=problem,
)

from pprint import pprint
pprint(output.model_dump())



[process] Key: 386160015.jpg#3r1e | Gold: entailment
[process] Premises: ['Two dogs playing on snow covered ground']
[process] Hypothesis: Dogs are playing on snow.
[process] Mode: full | Ablation: False
  [no-KB] Calling LangPro...
  [no-KB] Predicted: -
  [no-KB] Call failed.
{'ablation_results': None,
 'ablation_subsets': None,
 'essential_kb': None,
 'final_status': <ExperimentStatus.ERROR_NO_KB: 'error_no_kb'>,
 'fixed_by': None,
 'kb_details': None,
 'kb_filtered': None,
 'kb_raw': None,
 'pred_no_kb': <NLILabel.UNKNOWN: '-'>,
 'pred_with_kb': None,
 'pred_with_raw_kb': None,
 'problem': {'dataset': 'snli',
             'gold_label': <NLILabel.ENTAILMENT: 'entailment'>,
             'hypothesis': 'Dogs are playing on snow.',
             'id': '386160015.jpg#3r1e',
             'original_data': None,
             'premises': ['Two dogs playing on snow covered ground'],
             'split': 'dev'},
 'prover_calls': [{'ccg_terms': [],
                   'ccg_trees': [],
         

In [89]:
orchestration._run_ablation(
    prob=problem,
    kb_list=["isa_wn(dog, animal)"],
    gold_label=NLILabel.ENTAILMENT,
    verbose=True,
)

    

    [ablation] Testing subsets of size 1...
    [ablation] ['isa_wn(dog, animal)'] -> ERROR


([], [], {'isa_wn(dog, animal)': None})

In [94]:
from kbprojection import SNLILoader

orchestration.process_kb_examples(
    dataset=SNLILoader(),  # The dataset loader to use.
    config=None,   # Configuration for processing each problem.
    split=None,    # Dataset split to use.
    label_filter=None,    # Set of gold labels to consider.
    max_matches=None,     # Stop after yielding this many results (any outcome).
    max_checked=None,     # Maximum number of problems to check/process.
    problem_ids=None,     # Optional list of specific problem IDs to process.
    cache_dir=None        # Optional directory for caching results.
)
    


<generator object process_kb_examples at 0x12e452dc0>

In [99]:
from kbprojection.models import ExperimentResult


results = list(orchestration.process_kb_examples(
    dataset=SNLILoader(),
    split="dev",
    max_matches=5,
    max_checked=50,
))


[kb-processor] Processing split='dev', labels={'contradiction', 'entailment'}
[kb-processor] Config: mode=full, ablation=False
[kb-processor] Using random sampling.
[SNLILoader] Loaded 9842 problems from dev.

[kb-processor] #1 | Key: 3017220118.jpg#3r1e | Gold: entailment

[process] Key: 3017220118.jpg#3r1e | Gold: entailment
[process] Premises: ['Two women looking at the camera and a man looking away.']
[process] Hypothesis: A group of people.
[process] Mode: full | Ablation: False
  [no-KB] Calling LangPro...
  [no-KB] Predicted: -
  [no-KB] Call failed.

[kb-processor] #2 | Key: 2309779665.jpg#0r1e | Gold: entailment

[process] Key: 2309779665.jpg#0r1e | Gold: entailment
[process] Premises: ['the shadow silhouette of a woman standing near the water looking at a large attraction on the other side.']
[process] Hypothesis: A woman is casting a shadow.
[process] Mode: full | Ablation: False
  [no-KB] Calling LangPro...
  [no-KB] Predicted: -
  [no-KB] Call failed.

[kb-processor] #3 | 

In [107]:
from typing import Iterator

def count_to_three() -> Iterator[int]:
    yield 1
    yield 2
    yield 3


In [113]:
numbers = list(count_to_three())

numbers[2]


3